# 05 · Window functions (BigQuery)

A **window function** computes a value from a group of rows (the *window*) but **keeps every row**
(GROUP BY would squash them into one).

```
function() OVER (PARTITION BY <groups> ORDER BY <sort> ROWS BETWEEN ... )
```
- `PARTITION BY` = restart for each group (like GROUP BY, but rows stay)
- `ORDER BY` = the order inside the group (needed for ranking, running totals, LAG/LEAD)
- frame (`ROWS BETWEEN ...`) = which rows around the current row to use

You **cannot** use a window function in WHERE. Wrap it in a subquery/CTE and filter outside.
BigQuery (and Oracle 23ai) also has `QUALIFY`. Spark SQL has no QUALIFY.

**How to run:** Start each SQL cell with `%%bigquery`. Write tables as `shield.table_name`.
Each cell prints how much data it read. `%%bigquery dry` checks the size for free.

Tables: see [tables_guide.md](../../jarvis/tables_guide.md). Run the setup cell first (Shift+Enter).

> **Solutions.** Try the practice notebook first: [`bigquery/practice/05_window_functions.ipynb`](../../bigquery/practice/05_window_functions.ipynb)

In [ ]:
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "settings.py").exists())   # the repo folder
sys.path.insert(0, str(ROOT / "bigquery"))
%load_ext bq_helper

## Q1. ROW_NUMBER vs RANK vs DENSE_RANK  ·  🟢 easy

For department 2 (Sales), show `name`, `salary` and three columns ordered by salary (high first):
`rn` (ROW_NUMBER, ties broken by name), `rnk` (RANK), `drnk` (DENSE_RANK). Look at the 90,000 tie: what does each function do?

*Expected result: 6 rows · columns: `name`, `salary`, `rn`, `rnk`, `drnk`*

**How it works:** All three number rows by salary (high first) inside one window. They differ only on ties: ROW_NUMBER never repeats, RANK repeats then skips, DENSE_RANK repeats without gaps.

In [ ]:
%%bigquery
SELECT name, salary,
       ROW_NUMBER() OVER (ORDER BY salary DESC, name) AS rn,
       RANK()       OVER (ORDER BY salary DESC)       AS rnk,
       DENSE_RANK() OVER (ORDER BY salary DESC)       AS drnk
FROM shield.employees
WHERE dept_id = 2
ORDER BY rn

,name,salary,rn,rnk,drnk
0,Henry Clark,140000.000000000,1,1,1
1,Irene Lewis,90000.000000000,2,2,2
2,Jack Walker,90000.000000000,3,2,2
3,Karen Hall,90000.000000000,4,2,2
4,Leo Allen,70000.000000000,5,5,3
5,Mia Young,65000.000000000,6,6,4


**Why / traps:** Ties 90k ×3: ROW_NUMBER → 2,3,4 (always different). RANK → 2,2,2 then **5** (skips). DENSE_RANK → 2,2,2 then **3** (no gaps). Most asked window question.

## Q2. Top 2 salary levels per department  ·  🟡 medium

For each department (skip NULL dept_id) show employees whose salary is in the **top 2 different salary values** of that department. Show `dept_id`, `name`, `salary`.

*Expected result: 15 rows · columns: `dept_id`, `name`, `salary`*

**How it works:** 1) DENSE_RANK salaries inside each department (`PARTITION BY dept_id`). 2) Keep rank ≤ 2 in an outer query (or with QUALIFY in BigQuery).

In [ ]:
%%bigquery
SELECT dept_id, name, salary
FROM shield.employees
WHERE dept_id IS NOT NULL
QUALIFY DENSE_RANK() OVER (PARTITION BY dept_id ORDER BY salary DESC) <= 2
ORDER BY dept_id, salary DESC, name

,dept_id,name,salary
0,1,Alice Johnson,250000.000000000
1,1,Bob Smith,150000.000000000
2,2,Henry Clark,140000.000000000
3,2,Irene Lewis,90000.000000000
4,2,Jack Walker,90000.000000000
5,2,Karen Hall,90000.000000000
6,3,Nathan King,130000.000000000
7,3,Olivia Scott,85000.000000000
8,3,Paul Adams,85000.000000000
9,4,Rachel Nelson,135000.000000000


**Why / traps:** "Top N **people**" → ROW_NUMBER. "Top N **salaries**, keep ties" → DENSE_RANK. Ask which one!

`QUALIFY` filters on a window function, like HAVING does for GROUP BY. Without QUALIFY use the subquery way (works everywhere).

## Q3. Third highest salary  ·  🟡 medium

Show the **3rd highest different** salary in the company as `third_highest`.

*Expected result: 1 row · columns: `third_highest`*

**How it works:** DENSE_RANK over all salaries (no PARTITION BY); rank 3 = the third different salary. DISTINCT shows it once.

In [ ]:
%%bigquery
SELECT DISTINCT salary AS third_highest
FROM (SELECT salary, DENSE_RANK() OVER (ORDER BY salary DESC) AS drnk
      FROM shield.employees) t
WHERE drnk = 3

,third_highest
0,140000.000000000


**Why / traps:** DENSE_RANK + DISTINCT, because several people can share the 3rd salary. Other ways: `OFFSET 2` on distinct salaries; or a correlated COUNT(DISTINCT).

## Q4. Running total of revenue  ·  🟡 medium

For **completed** orders in 2024 (each order once), show `order_date`, `daily_revenue` and `running_total` (sum of all days so far in 2024).

*Expected result: 25 rows · columns: `order_date`, `daily_revenue`, `running_total`*

**How it works:** 1) One row per day (SUM per date). 2) `SUM(...) OVER (ORDER BY date ROWS UNBOUNDED PRECEDING → CURRENT ROW)` adds all days so far.

In [ ]:
%%bigquery
WITH daily AS (
    SELECT order_date, SUM(amount) AS daily_revenue
    FROM (SELECT DISTINCT order_id, order_date, amount, status FROM shield.orders) o
    WHERE status = 'completed'
      AND order_date >= DATE '2024-01-01' AND order_date < DATE '2025-01-01'
    GROUP BY order_date
)
SELECT order_date, daily_revenue,
       SUM(daily_revenue) OVER (ORDER BY order_date
                                ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS running_total
FROM daily
ORDER BY order_date

,order_date,daily_revenue,running_total
0,2024-01-08,82.000000000,82.000000000
1,2024-01-13,45.000000000,127.000000000
2,2024-01-21,80.000000000,207.000000000
3,2024-02-06,140.000000000,347.000000000
4,2024-02-07,62.000000000,409.000000000
5,2024-02-21,312.000000000,721.000000000
6,2024-03-05,306.000000000,1027.000000000
7,2024-03-13,40.000000000,1067.000000000
8,2024-03-22,30.000000000,1097.000000000
9,2024-04-10,165.000000000,1262.000000000


**Why / traps:** Aggregate to one row per day **first**, then run the window. A day with only NULL amounts gives NULL daily revenue; SUM over the window skips it.

## Q5. Salary vs department average  ·  🟢 easy

Show `name`, `dept_id`, `salary`, `dept_avg` (2 decimals) and `diff` = salary − dept_avg (2 decimals) for employees with a department.

*Expected result: 29 rows · columns: `name`, `dept_id`, `salary`, `dept_avg`, `diff`*

**How it works:** `AVG(salary) OVER (PARTITION BY dept_id)` puts the department average on every row; subtract it from the salary.

In [ ]:
%%bigquery
SELECT name, dept_id, salary,
       ROUND(AVG(salary) OVER (PARTITION BY dept_id), 2)          AS dept_avg,
       ROUND(salary - AVG(salary) OVER (PARTITION BY dept_id), 2) AS diff
FROM shield.employees
WHERE dept_id IS NOT NULL

,name,dept_id,salary,dept_avg,diff
0,Alice Johnson,1,250000.000000000,134285.710000000,115714.290000000
1,Bob Smith,1,150000.000000000,134285.710000000,15714.290000000
2,Carol White,1,120000.000000000,134285.710000000,-14285.710000000
3,David Brown,1,120000.000000000,134285.710000000,-14285.710000000
4,Eva Green,1,110000.000000000,134285.710000000,-24285.710000000
5,Frank Moore,1,95000.000000000,134285.710000000,-39285.710000000
6,Grace Lee,1,95000.000000000,134285.710000000,-39285.710000000
7,Henry Clark,2,140000.000000000,90833.330000000,49166.670000000
8,Irene Lewis,2,90000.000000000,90833.330000000,-833.330000000
9,Jack Walker,2,90000.000000000,90833.330000000,-833.330000000


**Why / traps:** With PARTITION BY and **no** ORDER BY, the window is the whole group: every row gets the group total/average.

## Q6. Share of department payroll  ·  🟢 easy

Show `name`, `dept_id`, `salary` and `pct_of_dept` = salary as % of the department's total salary, rounded to 1 decimal. Skip NULL dept_id.

*Expected result: 29 rows · columns: `name`, `dept_id`, `salary`, `pct_of_dept`*

**How it works:** `SUM(salary) OVER (PARTITION BY dept_id)` = department total on each row; salary / total × 100 = share.

In [ ]:
%%bigquery
SELECT name, dept_id, salary,
       ROUND(100 * salary / SUM(salary) OVER (PARTITION BY dept_id), 1) AS pct_of_dept
FROM shield.employees
WHERE dept_id IS NOT NULL

,name,dept_id,salary,pct_of_dept
0,Alice Johnson,1,250000.000000000,26.600000000
1,Bob Smith,1,150000.000000000,16.000000000
2,Carol White,1,120000.000000000,12.800000000
3,David Brown,1,120000.000000000,12.800000000
4,Eva Green,1,110000.000000000,11.700000000
5,Frank Moore,1,95000.000000000,10.100000000
6,Grace Lee,1,95000.000000000,10.100000000
7,Henry Clark,2,140000.000000000,25.700000000
8,Irene Lewis,2,90000.000000000,16.500000000
9,Jack Walker,2,90000.000000000,16.500000000


## Q7. Days since the previous order (LAG)  ·  🟡 medium

For each customer's orders (each order once), show `customer_id`, `order_id`, `order_date`, `prev_order_date` and `days_since_prev` (whole days). The first order of a customer has NULLs.

*Expected result: 104 rows · columns: `customer_id`, `order_id`, `order_date`, `prev_order_date`, `days_since_prev`*

**How it works:** `LAG(order_date)` over each customer's orders (sorted by date) gives the previous order date; subtract the two dates for the gap.

In [ ]:
%%bigquery
SELECT customer_id, order_id, order_date,
       LAG(order_date) OVER (PARTITION BY customer_id ORDER BY order_date, order_id) AS prev_order_date,
       DATE_DIFF(order_date,
                 LAG(order_date) OVER (PARTITION BY customer_id ORDER BY order_date, order_id),
                 DAY) AS days_since_prev
FROM (SELECT DISTINCT order_id, customer_id, order_date FROM shield.orders)

,customer_id,order_id,order_date,prev_order_date,days_since_prev
0,1,1026,2024-05-10,NaT,<NA>
1,2,1104,2025-12-28,NaT,<NA>
2,3,1003,2023-06-25,NaT,<NA>
3,4,1065,2025-05-04,NaT,<NA>
4,5,1057,2025-03-21,NaT,<NA>
5,6,1034,2024-08-04,NaT,<NA>
6,7,1002,2023-06-13,NaT,<NA>
7,8,1022,2024-03-31,NaT,<NA>
8,9,1056,2025-03-03,NaT,<NA>
9,9,1071,2025-06-16,2025-03-03,105


**Why / traps:** `LAG(col, n, default)` = value n rows **before**. `LEAD` = n rows **after**.

Date difference: `DATE_DIFF(later, earlier, DAY)`.

## Q8. Month-over-month change  ·  🔴 hard

Monthly revenue of completed orders (each order once). Show `month` (first day of the month), `revenue`, `prev_revenue`, and `pct_change` = (revenue − prev) / prev × 100, 1 decimal.

*Expected result: 27 rows · columns: `month`, `revenue`, `prev_revenue`, `pct_change`*

**How it works:** 1) CTE: revenue per month. 2) `LAG(revenue)` = last month's value. 3) (this − last) / last × 100.

In [ ]:
%%bigquery
WITH monthly AS (
    SELECT DATE_TRUNC(order_date, MONTH) AS month, SUM(amount) AS revenue
    FROM (SELECT DISTINCT order_id, order_date, amount, status FROM shield.orders) o
    WHERE status = 'completed'
    GROUP BY DATE_TRUNC(order_date, MONTH)
)
SELECT month, revenue,
       LAG(revenue) OVER (ORDER BY month) AS prev_revenue,
       ROUND(100 * (revenue - LAG(revenue) OVER (ORDER BY month))
                 / LAG(revenue) OVER (ORDER BY month), 1) AS pct_change
FROM monthly
ORDER BY month

,month,revenue,prev_revenue,pct_change
0,2023-03-01,135.000000000,None,None
1,2023-06-01,115.000000000,135.000000000,-14.800000000
2,2023-08-01,10.000000000,115.000000000,-91.300000000
3,2023-11-01,155.000000000,10.000000000,1450.000000000
4,2023-12-01,102.000000000,155.000000000,-34.200000000
5,2024-01-01,207.000000000,102.000000000,102.900000000
6,2024-02-01,514.000000000,207.000000000,148.300000000
7,2024-03-01,376.000000000,514.000000000,-26.800000000
8,2024-04-01,165.000000000,376.000000000,-56.100000000
9,2024-05-01,210.000000000,165.000000000,27.300000000


**Why / traps:** Months with no completed orders are **missing**, so LAG compares with an older month. To be exact, build a full month calendar first (topic 08). Division by zero: BigQuery has `SAFE_DIVIDE`; elsewhere use `NULLIF(prev, 0)`.

First day of month: `DATE_TRUNC(d, MONTH)`.

## Q9. 3-month moving average  ·  🟡 medium

Same monthly revenue. Show `month`, `revenue`, and `moving_avg_3` = average of this month and the 2 months before it (by row), 2 decimals.

*Expected result: 27 rows · columns: `month`, `revenue`, `moving_avg_3`*

**How it works:** Average over a frame of 3 rows: `ROWS BETWEEN 2 PRECEDING AND CURRENT ROW` = this month and the 2 rows before.

In [ ]:
%%bigquery
WITH monthly AS (
    SELECT DATE_TRUNC(order_date, MONTH) AS month, SUM(amount) AS revenue
    FROM (SELECT DISTINCT order_id, order_date, amount, status FROM shield.orders) o
    WHERE status = 'completed'
    GROUP BY DATE_TRUNC(order_date, MONTH)
)
SELECT month, revenue,
       ROUND(AVG(revenue) OVER (ORDER BY month ROWS BETWEEN 2 PRECEDING AND CURRENT ROW), 2) AS moving_avg_3
FROM monthly
ORDER BY month

,month,revenue,moving_avg_3
0,2023-03-01,135.000000000,135.000000000
1,2023-06-01,115.000000000,125.000000000
2,2023-08-01,10.000000000,86.670000000
3,2023-11-01,155.000000000,93.330000000
4,2023-12-01,102.000000000,89.000000000
5,2024-01-01,207.000000000,154.670000000
6,2024-02-01,514.000000000,274.330000000
7,2024-03-01,376.000000000,365.670000000
8,2024-04-01,165.000000000,351.670000000
9,2024-05-01,210.000000000,250.330000000


**Why / traps:** `ROWS BETWEEN 2 PRECEDING AND CURRENT ROW` = 3 rows. The first 2 months average fewer rows.

## Q10. Salary quartiles (NTILE)  ·  🟢 easy

Split all employees into 4 groups by salary (highest first): `name`, `salary`, `quartile` (1–4). Break ties by `emp_id`.

*Expected result: 30 rows · columns: `name`, `salary`, `quartile`*

**How it works:** `NTILE(4)` sorts the rows and cuts them into 4 groups of (almost) the same size.

In [ ]:
%%bigquery
SELECT name, salary,
       NTILE(4) OVER (ORDER BY salary DESC, emp_id) AS quartile
FROM shield.employees

,name,salary,quartile
0,Alice Johnson,250000.000000000,1
1,Bob Smith,150000.000000000,1
2,Henry Clark,140000.000000000,1
3,Rachel Nelson,135000.000000000,1
4,Nathan King,130000.000000000,1
5,Carol White,120000.000000000,1
6,David Brown,120000.000000000,1
7,Yara Campbell,115000.000000000,1
8,Eva Green,110000.000000000,2
9,Victor Roberts,105000.000000000,2


**Why / traps:** 30 rows / 4 groups: the first 2 groups get 8 rows, the last 2 get 7.

## Q11. Top earner on every row (FIRST_VALUE)  ·  🟡 medium

Show `name`, `dept_id`, `salary` and `top_earner` = the name of the highest-paid person in the same department (ties: first by name). Skip NULL dept_id.

*Expected result: 29 rows · columns: `name`, `dept_id`, `salary`, `top_earner`*

**How it works:** `FIRST_VALUE(name)` over each department sorted by salary high first returns the top earner's name on every row.

In [ ]:
%%bigquery
SELECT name, dept_id, salary,
       FIRST_VALUE(name) OVER (PARTITION BY dept_id ORDER BY salary DESC, name) AS top_earner
FROM shield.employees
WHERE dept_id IS NOT NULL

,name,dept_id,salary,top_earner
0,Alice Johnson,1,250000.000000000,Alice Johnson
1,Bob Smith,1,150000.000000000,Alice Johnson
2,Carol White,1,120000.000000000,Alice Johnson
3,David Brown,1,120000.000000000,Alice Johnson
4,Eva Green,1,110000.000000000,Alice Johnson
5,Frank Moore,1,95000.000000000,Alice Johnson
6,Grace Lee,1,95000.000000000,Alice Johnson
7,Henry Clark,2,140000.000000000,Henry Clark
8,Irene Lewis,2,90000.000000000,Henry Clark
9,Jack Walker,2,90000.000000000,Henry Clark


**Why / traps:** Trap: `LAST_VALUE(...) OVER (ORDER BY ...)` gives the **current** row, because the default frame ends at the current row. Add `ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING`.

## Q12. Latest order per customer  ·  🟡 medium

For each customer, show their latest order: `customer_id`, `order_id`, `order_date`, `amount`. If two orders are on the same day, take the bigger `order_id`. One row per customer.

*Expected result: 32 rows · columns: `customer_id`, `order_id`, `order_date`, `amount`*

**How it works:** ROW_NUMBER per customer, newest first (`order_date DESC, order_id DESC`), then keep `rn = 1` = exactly one latest row per customer.

In [ ]:
%%bigquery
SELECT customer_id, order_id, order_date, amount
FROM shield.orders
WHERE TRUE
QUALIFY ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY order_date DESC, order_id DESC) = 1

,customer_id,order_id,order_date,amount
0,1,1026,2024-05-10,40.000000000
1,2,1104,2025-12-28,135.000000000
2,3,1003,2023-06-25,70.000000000
3,4,1065,2025-05-04,40.000000000
4,5,1057,2025-03-21,90.000000000
5,6,1034,2024-08-04,42.000000000
6,7,1002,2023-06-13,45.000000000
7,8,1022,2024-03-31,225.000000000
8,9,1071,2025-06-16,12.000000000
9,10,1093,2025-09-28,69.000000000


**Why / traps:** ROW_NUMBER = exactly one row per group. It also removes the duplicate rows for free. Always add a tie-breaker to ORDER BY, so the result is the same every run.

`WHERE TRUE` is a habit: older BigQuery needed a WHERE/GROUP BY/HAVING next to QUALIFY.

## Q13. ROWS vs RANGE (ties in a running total)  ·  🔴 hard

Department 2 employees ordered by salary (low → high). Show `name`, `salary`, `rows_total` (frame ROWS ... CURRENT ROW, ties broken by emp_id) and `range_total` (frame RANGE ... CURRENT ROW). Why are they different for the 90,000 tie?

*Expected result: 6 rows · columns: `name`, `salary`, `rows_total`, `range_total`*

**How it works:** ROWS adds one physical row at a time. RANGE adds **all rows with the same salary** at once, so tied rows show the same total.

In [ ]:
%%bigquery
SELECT name, salary,
       SUM(salary) OVER (ORDER BY salary, emp_id ROWS  BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS rows_total,
       SUM(salary) OVER (ORDER BY salary         RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS range_total
FROM shield.employees
WHERE dept_id = 2
ORDER BY salary, emp_id

,name,salary,rows_total,range_total
0,Mia Young,65000.000000000,65000.000000000,65000.000000000
1,Leo Allen,70000.000000000,135000.000000000,135000.000000000
2,Irene Lewis,90000.000000000,225000.000000000,405000.000000000
3,Jack Walker,90000.000000000,315000.000000000,405000.000000000
4,Karen Hall,90000.000000000,405000.000000000,405000.000000000
5,Henry Clark,140000.000000000,545000.000000000,545000.000000000


**Why / traps:** ROWS counts physical rows. RANGE includes **all rows with the same ORDER BY value** (peers). The default frame with ORDER BY is RANGE: that is why running totals jump on ties.